# FusionCore v0 — Phase 5b: Official Test Set Inference — XGBoost

**Notebook:** `05b_XGBoost_Inference.ipynb`  
**Phase:** 5 of 5 (Part B)  
**Objective:** XGBoost inference and SHAP analysis on the Official Test Set.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1 — Environment Setup (Run First)
# ══════════════════════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 2 — Dependency Installation
# ══════════════════════════════════════════════════════════════════════════════

%%capture
!pip install shap

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 3 — Imports, Constants & Palette
# ══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import os
import gc

DRIVE_ROOT     = Path('/content/drive/MyDrive/PI')
OUTPUTS_DIR    = DRIVE_ROOT / 'FusionCore' / 'v0' / 'outputs'
CHECKPOINT_DIR = DRIVE_ROOT / 'FusionCore' / 'v0' / 'checkpoints'

RUL_CAP      = 125
RANDOM_STATE = 42
UNIT_KEY     = ['subset_origin', 'unit_id']
CMAPSS_SUBSETS = ['FD001', 'FD002', 'FD003', 'FD004']

np.random.seed(RANDOM_STATE)

FC_DARK_BLUE = '#0D1B2A'; FC_NAVY = '#1B3A5C'; FC_ORANGE = '#D96A1B'
FC_DEEP_RED = '#9B1B30'; FC_STEEL = '#4A6274'; FC_CHARCOAL = '#2D2D2D'

plt.rcParams.update({
    'figure.figsize': (14, 5), 'figure.dpi': 150, 'savefig.dpi': 300,
    'savefig.bbox': 'tight', 'axes.spines.top': False, 'axes.spines.right': False,
})

from sklearn.metrics import mean_squared_error, mean_absolute_error

def compute_nasa_score(y_true, y_pred):
    d = y_pred - y_true
    return float(np.sum(np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 4 — Load Test Set Artefacts from 05a
# ══════════════════════════════════════════════════════════════════════════════

X_test    = pd.read_parquet(OUTPUTS_DIR / 'X_test.parquet')
X_test_nn = pd.read_parquet(OUTPUTS_DIR / 'X_test_nn.parquet')
meta_test = pd.read_parquet(OUTPUTS_DIR / 'meta_test.parquet')
y_test_df = pd.read_parquet(OUTPUTS_DIR / 'y_test.parquet')

feature_manifest = pd.read_csv(OUTPUTS_DIR / 'feature_manifest.csv')
feature_names_91 = list(feature_manifest['feature'])
nn_feature_names = joblib.load(OUTPUTS_DIR / 'nn_feature_names.pkl')

# Build last-cycle index and engine metadata.
test_last_idx = meta_test.groupby(UNIT_KEY)['cycle'].idxmax()
test_last = meta_test.loc[test_last_idx].reset_index(drop=True)
test_engine_meta = test_last[['subset_origin', 'unit_id']].merge(
    y_test_df, on=['subset_origin', 'unit_id'], how='left'
)
y_test = test_engine_meta['RUL'].values

print(f'X_test: {X_test.shape}, X_test_nn: {X_test_nn.shape}')
print(f'Test engines: {len(y_test)}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 5 — Load XGBoost Checkpoint & Inference
# ══════════════════════════════════════════════════════════════════════════════

import xgboost as xgb

xgb_model = xgb.XGBRegressor()
xgb_model.load_model(str(CHECKPOINT_DIR / 'xgb_baseline.json'))
print('✔ XGBoost checkpoint loaded.')

# Last-cycle features (91 features).
X_test_last = X_test.loc[test_last_idx].reset_index(drop=True)
X_test_last = X_test_last[feature_names_91]

y_pred_xgb = xgb_model.predict(X_test_last)

rmse = float(np.sqrt(mean_squared_error(y_test, y_pred_xgb)))
mae  = float(mean_absolute_error(y_test, y_pred_xgb))
nasa = compute_nasa_score(y_test, y_pred_xgb)

print(f'XGBoost Official Test Set Results')
print(f'  RMSE:       {rmse:.4f}')
print(f'  MAE:        {mae:.4f}')
print(f'  NASA Score: {nasa:,.1f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 6 — SHAP Analysis (Official Test Set)
# ══════════════════════════════════════════════════════════════════════════════

import shap

explainer = shap.TreeExplainer(xgb_model)
SHAP_SAMPLE = min(1000, len(X_test_last))
X_shap_sample = X_test_last.sample(n=SHAP_SAMPLE, random_state=RANDOM_STATE)
shap_values_test = explainer.shap_values(X_shap_sample)

fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values_test, X_shap_sample,
                  feature_names=feature_names_91, max_display=30, show=False)
plt.title('SHAP Feature Importance — XGBoost (Official Test Set)',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Top features and engineered feature check.
mean_abs_shap = np.abs(shap_values_test).mean(axis=0)
shap_ranking_test = pd.Series(mean_abs_shap, index=feature_names_91).sort_values(ascending=False)
print('\nTop 10 Features (Official Test Set):')
for i, (feat, val) in enumerate(shap_ranking_test.head(10).items(), 1):
    print(f'  {i:>2}. {feat:<30} {val:.4f}')

engineered = ['CPR', 'E_thermal', 'EGT_drift', 's4_cumfatigue', 's7_cumfatigue', 's9_cumfatigue']
top20 = set(shap_ranking_test.head(20).index)
eng_in_top20 = [f for f in engineered if f in top20]
shap_check = len(eng_in_top20) >= 2
print(f'\nEngineered features in top 20: {eng_in_top20}')
print(f'SHAP dominance check: {"PASS" if shap_check else "INVESTIGATE"}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 7 — Save 05b Outputs
# ══════════════════════════════════════════════════════════════════════════════

joblib.dump({
    'y_pred_xgb': y_pred_xgb,
    'y_test': y_test,
    'test_engine_meta': test_engine_meta,
    'shap_ranking_test': shap_ranking_test.to_dict(),
    'shap_check': shap_check,
    'rmse': rmse, 'mae': mae, 'nasa_score': nasa,
}, OUTPUTS_DIR / 'phase5b_xgb_predictions.pkl')

print('Phase 5b outputs persisted.')
print(f'✔ Notebook 05b complete.')